# 01 Data Loading and Inspection

This notebook loads the SCANIA Component X dataset and performs an initial inspection of the operational readouts, time-to-event labels, and vehicle specification data.

The purpose of this notebook is to get acquainted with the dataset, verify its structure before preprocessing, feature engineering, sequence construction and modelling.

## Inputs

- `data/raw/train_operational_readouts.csv`
- `data/raw/train_tte.csv`
- `data/raw/train_specifications.csv`

## Outputs

This notebook does not generate final modelling files. It is mainly used for inspection and validation.

In [ ]:
# Set project paths and enable imports from the src directory

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

print(f"Notebook location: {Path.cwd()}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Source directory: {SRC_DIR}")

In [ ]:
# Verify that required raw data files are available

from config import OPERATIONAL_FILE, SPECIFICATIONS_FILE, TTE_FILE

required_files = {
    "Operational readouts": OPERATIONAL_FILE,
    "Vehicle specifications": SPECIFICATIONS_FILE,
    "Time-to-event labels": TTE_FILE,
}

for name, path in required_files.items():
    print(f"{name}: {path}")
    print(f"Exists: {path.exists()}")
    print()

In [ ]:
# Load the training datasets
from config import OPERATIONAL_FILE, SPECIFICATIONS_FILE, TTE_FILE
from data_loading import load_raw_training_data, inspect_dataframe

operational_df, spec_df, tte_df = load_raw_training_data(
    OPERATIONAL_FILE,
    SPECIFICATIONS_FILE,
    TTE_FILE
)

In [ ]:
# Basic inspection of the datasets: shapes, column names, missing values
inspect_dataframe(operational_df, "Operational")
inspect_dataframe(spec_df, "Specifications")
inspect_dataframe(tte_df, "TTE")

In [ ]:
# Sort operational readouts by vehicle_id and time_step
from quality_checks import sort_operational_data

operational_sorted_df = sort_operational_data(operational_df)
operational_sorted_df.head(10)


In [ ]:
# Retrieve a subset of the sorted dataset to check how the sorting has worked.
operational_sorted_df.loc[operational_sorted_df["vehicle_id"] == 0, ["vehicle_id", "time_step"]].head(20)

In [ ]:
# Check if there are duplicates present in the dataset
from quality_checks import check_duplicate_vehicle_time_pairs

duplicates_df = check_duplicate_vehicle_time_pairs(operational_sorted_df)
duplicates_df.head(20)

In [ ]:
# check for monotonicity, if the time_step is monotonically increasing
from quality_checks import check_time_monotonicity

monotonicity_violations_df = check_time_monotonicity(operational_sorted_df)
monotonicity_violations_df.head(20)

In [ ]:
# Retrieve information about time gaps
from quality_checks import (
    compute_time_gap_summary,
    summarize_time_gaps,
    find_large_time_gaps,
)
# Create a dataset of time gaps per row 
operational_gap_df = compute_time_gap_summary(operational_sorted_df)
# Inspect time gaps statistics
gap_summary = summarize_time_gaps(operational_gap_df)
# Look at the largest time gaps in the dataset, exceeding a certain threshold
large_gaps_df = find_large_time_gaps(operational_gap_df, threshold=50)
large_gaps_df[["vehicle_id", "time_step", "prev_time_step", "time_gap"]].head(20)